In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Load both files ---
sensor_df = pd.read_csv('session_20260630_205845.csv')
phase_df = pd.read_csv('session_20260630_210910.csv')

# --- Convert to seconds relative to sensor session start ---
t0 = sensor_df['wallclock_ms'].iloc[0]
sensor_df['t_sec'] = (sensor_df['wallclock_ms'] - t0) / 1000
phase_df['t_sec'] = (phase_df['event_timestamp_ms'] - t0) / 1000

print(f"Sensor session duration: {sensor_df['t_sec'].iloc[-1]:.1f}s")
print(f"Phase log entries:\n{phase_df[['phase_name','t_sec']]}")

In [ ]:
# --- Plot all three sensors with phase boundaries overlaid ---
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(sensor_df['t_sec'], sensor_df['eda_conductance_us'])
axes[0].set_ylabel('EDA (µS)')
axes[0].set_title('Automated-Timer Calibration Session')

axes[1].plot(sensor_df['t_sec'], sensor_df['ppg_ir'], alpha=0.7)
axes[1].set_ylabel('PPG (IR)')

acc_mag = (sensor_df['acc_x']**2 + sensor_df['acc_y']**2 + sensor_df['acc_z']**2) ** 0.5
axes[2].plot(sensor_df['t_sec'], acc_mag)
axes[2].set_ylabel('Accel Magnitude')
axes[2].set_xlabel('Time (seconds)')

# Overlay phase boundaries -- these are now exact, not manually estimated
for ax in axes:
    for _, row in phase_df.iterrows():
        ax.axvline(row['t_sec'], color='gray', linestyle='--', alpha=0.6)
        if row['event_type'] == 'phase_start':
            ax.text(row['t_sec'], ax.get_ylim()[1], row['phase_name'],
                    rotation=90, fontsize=8, va='top')

plt.tight_layout()
plt.show()

In [ ]:
print(sensor_df['wallclock_ms'].diff().describe())

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# --- Load data (reuse from before) ---
sensor_df = pd.read_csv('session_20260630_205845.csv')
phase_df = pd.read_csv('session_20260630_210910.csv')

t0 = sensor_df['wallclock_ms'].iloc[0]
sensor_df['t_sec'] = (sensor_df['wallclock_ms'] - t0) / 1000
phase_df['t_sec'] = (phase_df['event_timestamp_ms'] - t0) / 1000

# --- Build phase windows (start, end) from the phase log ---
phase_starts = phase_df[phase_df['event_type'] == 'phase_start'].reset_index(drop=True)
phase_windows = []
for i in range(len(phase_starts)):
    name = phase_starts.loc[i, 'phase_name']
    start = phase_starts.loc[i, 't_sec']
    # end = next phase's start, or session_end for the last one
    if i + 1 < len(phase_starts):
        end = phase_starts.loc[i + 1, 't_sec']
    else:
        end = phase_df[phase_df['event_type'] == 'session_end']['t_sec'].iloc[0]
    phase_windows.append((name, start, end))

print("Phase windows (seconds):")
for name, start, end in phase_windows:
    print(f"  {name:20s} {start:7.1f} -> {end:7.1f}  ({end-start:.0f}s)")

In [ ]:
# --- Compute EDA stats per phase ---
acc_mag = np.sqrt(sensor_df['acc_x']**2 + sensor_df['acc_y']**2 + sensor_df['acc_z']**2)
sensor_df['acc_mag'] = acc_mag

results = []
for name, start, end in phase_windows:
    window = sensor_df[(sensor_df['t_sec'] >= start) & (sensor_df['t_sec'] < end)]

    if len(window) < 2:
        continue

    eda_mean = window['eda_conductance_us'].mean()
    eda_std = window['eda_conductance_us'].std()

    # Linear trend (slope) across the window -- positive = rising, negative = falling
    slope, intercept, r_value, p_value, std_err = stats.linregress(
        window['t_sec'], window['eda_conductance_us']
    )

    acc_mean = window['acc_mag'].mean()
    acc_std = window['acc_mag'].std()  # variability is more informative than mean for movement

    results.append({
        'phase': name,
        'n_samples': len(window),
        'eda_mean': eda_mean,
        'eda_std': eda_std,
        'eda_slope_per_sec': slope,
        'eda_trend_p': p_value,
        'acc_mean': acc_mean,
        'acc_std': acc_std,
    })

results_df = pd.DataFrame(results)
pd.set_option('display.float_format', '{:.4f}'.format)
print(results_df.to_string(index=False))

In [ ]:
# --- Direct comparison: Arithmetic vs. the two Rest phases ---
arith = sensor_df[(sensor_df['t_sec'] >= phase_windows[2][1]) & 
                   (sensor_df['t_sec'] < phase_windows[2][2])]['eda_conductance_us']

rest_before = sensor_df[(sensor_df['t_sec'] >= phase_windows[1][1]) & 
                         (sensor_df['t_sec'] < phase_windows[1][2])]['eda_conductance_us']

rest_after = sensor_df[(sensor_df['t_sec'] >= phase_windows[3][1]) & 
                        (sensor_df['t_sec'] < phase_windows[3][2])]['eda_conductance_us']

print(f"Rest (before arithmetic):  mean={rest_before.mean():.4f}  std={rest_before.std():.4f}")
print(f"Mental Arithmetic:         mean={arith.mean():.4f}  std={arith.std():.4f}")
print(f"Rest/Recovery (after):     mean={rest_after.mean():.4f}  std={rest_after.std():.4f}")

# Non-parametric comparison since these are small, non-independent samples within one session
u_stat_before, p_before = stats.mannwhitneyu(arith, rest_before, alternative='two-sided')
u_stat_after, p_after = stats.mannwhitneyu(arith, rest_after, alternative='two-sided')

print(f"\nArithmetic vs Rest-before: Mann-Whitney p = {p_before:.4f}")
print(f"Arithmetic vs Rest-after:  Mann-Whitney p = {p_after:.4f}")
print("\n(Note: these p-values are exploratory only -- one session, not independent")
print(" samples in the formal statistical sense. Useful for direction/magnitude,")
print(" not for a real hypothesis test.)")

In [ ]:
# --- Specifically check the Deliberate Movement phase accelerometer response ---
movement = sensor_df[(sensor_df['t_sec'] >= phase_windows[5][1]) & 
                      (sensor_df['t_sec'] < phase_windows[5][2])]['acc_mag']

baseline_acc = sensor_df[(sensor_df['t_sec'] >= phase_windows[0][1]) & 
                          (sensor_df['t_sec'] < phase_windows[0][2])]['acc_mag']

print(f"True Baseline accel std:      {baseline_acc.std():.4f}")
print(f"Deliberate Movement accel std: {movement.std():.4f}")
print(f"Ratio (movement/baseline):     {movement.std() / baseline_acc.std():.2f}x")
print("\nIf this ratio is close to 1, the movement phase likely didn't register")
print("meaningfully above baseline -- worth checking against what you actually did.")

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# --- Load data (reuse from before) ---
sensor_df = pd.read_csv('session_20260630_205845.csv')
phase_df = pd.read_csv('session_20260630_210910.csv')

t0 = sensor_df['wallclock_ms'].iloc[0]
sensor_df['t_sec'] = (sensor_df['wallclock_ms'] - t0) / 1000
phase_df['t_sec'] = (phase_df['event_timestamp_ms'] - t0) / 1000

sensor_df['acc_mag'] = np.sqrt(
    sensor_df['acc_x']**2 + sensor_df['acc_y']**2 + sensor_df['acc_z']**2
)

# --- Rebuild phase windows ---
phase_starts = phase_df[phase_df['event_type'] == 'phase_start'].reset_index(drop=True)
phase_windows = []
for i in range(len(phase_starts)):
    name = phase_starts.loc[i, 'phase_name']
    start = phase_starts.loc[i, 't_sec']
    end = (phase_starts.loc[i + 1, 't_sec'] if i + 1 < len(phase_starts)
           else phase_df[phase_df['event_type'] == 'session_end']['t_sec'].iloc[0])
    phase_windows.append((name, start, end))

In [ ]:
# --- Flag motion artifacts using a rolling local baseline ---
# Using the WHOLE-SESSION mean/std here, matching your existing pipeline
# convention (ACC std > mean + 2SD) from the 6/29 artifact-exclusion work.
acc_mean_global = sensor_df['acc_mag'].mean()
acc_std_global = sensor_df['acc_mag'].std()
artifact_threshold = acc_mean_global + 2 * acc_std_global

sensor_df['is_artifact'] = sensor_df['acc_mag'] > artifact_threshold

print(f"Global accel mean: {acc_mean_global:.4f}, std: {acc_std_global:.4f}")
print(f"Artifact threshold: {artifact_threshold:.4f}")
print(f"Total samples: {len(sensor_df)}")
print(f"Flagged as artifact: {sensor_df['is_artifact'].sum()} "
      f"({100*sensor_df['is_artifact'].mean():.1f}%)")

In [ ]:
# --- Recompute per-phase stats, excluding flagged samples ---
results = []
for name, start, end in phase_windows:
    window = sensor_df[(sensor_df['t_sec'] >= start) & (sensor_df['t_sec'] < end)]
    clean = window[~window['is_artifact']]

    n_excluded = len(window) - len(clean)
    pct_excluded = 100 * n_excluded / len(window) if len(window) > 0 else 0

    if len(clean) < 2:
        continue

    slope, intercept, r_value, p_value, std_err = stats.linregress(
        clean['t_sec'], clean['eda_conductance_us']
    )

    results.append({
        'phase': name,
        'n_total': len(window),
        'n_excluded': n_excluded,
        'pct_excluded': pct_excluded,
        'eda_mean_clean': clean['eda_conductance_us'].mean(),
        'eda_std_clean': clean['eda_conductance_us'].std(),
        'eda_slope_clean': slope,
    })

results_df = pd.DataFrame(results)
pd.set_option('display.float_format', '{:.4f}'.format)
print(results_df.to_string(index=False))

In [ ]:
# --- Direct arithmetic vs rest comparison, artifact-excluded ---
def clean_phase_eda(idx):
    name, start, end = phase_windows[idx]
    window = sensor_df[(sensor_df['t_sec'] >= start) & (sensor_df['t_sec'] < end)]
    return window[~window['is_artifact']]['eda_conductance_us']

rest_before = clean_phase_eda(1)
arith = clean_phase_eda(2)
rest_after = clean_phase_eda(3)

print(f"Rest (before), n={len(rest_before)}:  mean={rest_before.mean():.4f}")
print(f"Arithmetic, n={len(arith)}:            mean={arith.mean():.4f}")
print(f"Rest/Recovery (after), n={len(rest_after)}: mean={rest_after.mean():.4f}")

u_before, p_before = stats.mannwhitneyu(arith, rest_before, alternative='two-sided')
u_after, p_after = stats.mannwhitneyu(arith, rest_after, alternative='two-sided')
print(f"\nArithmetic vs Rest-before (artifact-excluded): p = {p_before:.4f}")
print(f"Arithmetic vs Rest-after (artifact-excluded):  p = {p_after:.4f}")

In [ ]:
# --- Recompute artifact threshold using ONLY the labeled session, 
#     excluding pre-session setup noise ---
session_start = phase_windows[0][1]  # True Baseline start
session_only = sensor_df[sensor_df['t_sec'] >= session_start]

acc_mean_session = session_only['acc_mag'].mean()
acc_std_session = session_only['acc_mag'].std()
artifact_threshold_session = acc_mean_session + 2 * acc_std_session

print(f"Session-only accel mean: {acc_mean_session:.4f}, std: {acc_std_session:.4f}")
print(f"Session-only threshold:   {artifact_threshold_session:.4f}")
print(f"(compare to old global threshold: 10.5506 -- inflated by setup spikes)")

sensor_df['is_artifact_instant'] = sensor_df['acc_mag'] > artifact_threshold_session
print(f"\nFlagged with session-only threshold: {sensor_df['is_artifact_instant'].sum()} "
      f"({100*sensor_df['is_artifact_instant'].mean():.2f}%)")

In [ ]:
# --- Windowed exclusion: expand each flagged spike to cover the 
#     likely EDA decay tail (e.g. 3s before, 15s after) ---
PRE_WINDOW_SEC = 3
POST_WINDOW_SEC = 15

sensor_df['is_artifact'] = False
spike_times = sensor_df.loc[sensor_df['is_artifact_instant'], 't_sec'].values

for spike_t in spike_times:
    mask = (sensor_df['t_sec'] >= spike_t - PRE_WINDOW_SEC) & \
           (sensor_df['t_sec'] <= spike_t + POST_WINDOW_SEC)
    sensor_df.loc[mask, 'is_artifact'] = True

print(f"Total flagged after windowing: {sensor_df['is_artifact'].sum()} "
      f"({100*sensor_df['is_artifact'].mean():.1f}%)")

In [ ]:
# --- Rerun per-phase stats with the corrected, windowed exclusion ---
results = []
for name, start, end in phase_windows:
    window = sensor_df[(sensor_df['t_sec'] >= start) & (sensor_df['t_sec'] < end)]
    clean = window[~window['is_artifact']]
    n_excluded = len(window) - len(clean)
    pct_excluded = 100 * n_excluded / len(window) if len(window) > 0 else 0

    if len(clean) < 2:
        results.append({'phase': name, 'n_total': len(window), 
                         'n_excluded': n_excluded, 'pct_excluded': pct_excluded,
                         'eda_mean_clean': None})
        continue

    results.append({
        'phase': name,
        'n_total': len(window),
        'n_excluded': n_excluded,
        'pct_excluded': pct_excluded,
        'eda_mean_clean': clean['eda_conductance_us'].mean(),
        'eda_std_clean': clean['eda_conductance_us'].std(),
    })

results_df = pd.DataFrame(results)
pd.set_option('display.float_format', '{:.4f}'.format)
print(results_df.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import find_peaks

sensor_df = pd.read_csv('session_20260630_205845.csv')
t0 = sensor_df['wallclock_ms'].iloc[0]
sensor_df['t_sec'] = (sensor_df['wallclock_ms'] - t0) / 1000

# Restrict to the labeled session (skip pre-session setup with PPG dropout)
session_start = 621.284  # True Baseline start, from earlier
session_df = sensor_df[sensor_df['t_sec'] >= session_start].copy()

# Rough peak detection on IR channel per phase window
def estimate_hr(window_df, min_distance_sec=0.4):
    # min_distance_sec ~0.4s caps HR at 150bpm, prevents double-counting noise
    signal = window_df['ppg_ir'].values
    times = window_df['t_sec'].values
    if len(signal) < 10:
        return None, 0

    # crude smoothing first, given low sample rate
    smoothed = pd.Series(signal).rolling(3, center=True, min_periods=1).mean().values

    peaks, _ = find_peaks(smoothed, distance=2)  # distance in samples, not seconds, at 4Hz
    if len(peaks) < 3:
        return None, len(peaks)

    peak_times = times[peaks]
    ibi = np.diff(peak_times)  # inter-beat intervals in seconds
    ibi = ibi[(ibi > 0.4) & (ibi < 1.5)]  # keep only physiologically plausible (40-150bpm)
    if len(ibi) < 2:
        return None, len(peaks)

    hr_bpm = 60 / np.median(ibi)
    return hr_bpm, len(peaks)

for name, start, end in phase_windows:
    window = session_df[(session_df['t_sec'] >= start) & (session_df['t_sec'] < end)]
    hr, n_peaks = estimate_hr(window)
    hr_str = f"{hr:.1f} bpm" if hr else "insufficient/unreliable"
    print(f"{name:20s}  est. HR: {hr_str:20s}  (n_peaks detected: {n_peaks})")

In [ ]:
for name, start, end in phase_windows:
    window = session_df[(session_df['t_sec'] >= start) & (session_df['t_sec'] < end)]
    print(f"{name:20s}  PPG IR mean: {window['ppg_ir'].mean():8.0f}  "
          f"std: {window['ppg_ir'].std():7.1f}")